# Digital Career Counselor — Complete ML Project
### Run each cell top to bottom. That's it!
**Steps covered:** Setup → Dataset → Explore → Train → Predict → Visualize

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

print('All libraries loaded successfully!')

## Step 2 — Generate the Dataset (500 students)

In [ ]:
np.random.seed(42)
N = 500

CAREERS = [
    'Sports Analytics',
    'Data Science / Engineering',
    'Medicine / Research',
    'Professional Athlete / Coach',
    'Arts & Humanities',
    'Business / Management'
]

SPORTS    = ['Football', 'Cricket', 'Swimming', 'Athletics', 'Basketball', 'Badminton', 'None']
SPORT_ENC = {s: i for i, s in enumerate(SPORTS)}

def mark(base, std=12):
    return int(np.clip(np.random.normal(base, std), 30, 100))

rows = []
for _ in range(N):
    career = CAREERS[np.random.randint(0, 6)]

    if career == 'Sports Analytics':
        math, physics, chemistry, biology, english = mark(82), mark(72), mark(60), mark(60), mark(68)
        attendance = int(np.clip(np.random.normal(80, 8), 50, 100))
        focus      = int(np.clip(np.random.normal(55, 10), 20, 120))
        sport      = np.random.choice(['Football', 'Cricket', 'Basketball', 'Athletics'])
        skill      = int(np.clip(np.random.normal(8, 1), 6, 10))

    elif career == 'Data Science / Engineering':
        math, physics, chemistry, biology, english = mark(90), mark(84), mark(70), mark(55), mark(65)
        attendance = int(np.clip(np.random.normal(85, 7), 55, 100))
        focus      = int(np.clip(np.random.normal(75, 12), 30, 120))
        sport      = np.random.choice(['None', 'Badminton', 'Swimming', 'None'])
        skill      = int(np.clip(np.random.normal(4, 2), 1, 7))

    elif career == 'Medicine / Research':
        math, physics, chemistry, biology, english = mark(72), mark(70), mark(88), mark(92), mark(72)
        attendance = int(np.clip(np.random.normal(90, 5), 65, 100))
        focus      = int(np.clip(np.random.normal(85, 10), 40, 120))
        sport      = np.random.choice(['None', 'Swimming', 'Badminton', 'None'])
        skill      = int(np.clip(np.random.normal(3, 2), 1, 6))

    elif career == 'Professional Athlete / Coach':
        math, physics, chemistry, biology, english = mark(55), mark(52), mark(50), mark(60), mark(58)
        attendance = int(np.clip(np.random.normal(72, 10), 40, 100))
        focus      = int(np.clip(np.random.normal(45, 10), 20, 90))
        sport      = np.random.choice(['Football', 'Cricket', 'Basketball', 'Athletics', 'Swimming'])
        skill      = int(np.clip(np.random.normal(9, 0.8), 7, 10))

    elif career == 'Arts & Humanities':
        math, physics, chemistry, biology, english = mark(52), mark(50), mark(50), mark(55), mark(88)
        attendance = int(np.clip(np.random.normal(75, 10), 45, 100))
        focus      = int(np.clip(np.random.normal(60, 15), 20, 110))
        sport      = np.random.choice(['None', 'Badminton', 'None', 'None'])
        skill      = int(np.clip(np.random.normal(3, 2), 1, 6))

    else:  # Business / Management
        math, physics, chemistry, biology, english = mark(75), mark(60), mark(58), mark(55), mark(80)
        attendance = int(np.clip(np.random.normal(82, 8), 50, 100))
        focus      = int(np.clip(np.random.normal(65, 12), 25, 110))
        sport      = np.random.choice(['Cricket', 'Football', 'Badminton', 'None'])
        skill      = int(np.clip(np.random.normal(5, 2), 1, 8))

    rows.append({
        'Math': math, 'Physics': physics, 'Chemistry': chemistry,
        'Biology': biology, 'English': english,
        'Attendance_Pct': attendance,
        'Focus_Duration_Min': focus,
        'Sport_Type': sport,
        'Sport_Type_Enc': SPORT_ENC[sport],
        'Sport_Skill_Level': skill,
        'Career_Label': career
    })

df = pd.DataFrame(rows)
df.to_csv('student_career_dataset.csv', index=False)
print(f'Dataset created: {df.shape[0]} rows, {df.shape[1]} columns')
print('Career distribution:')
print(df['Career_Label'].value_counts())
df.head()

## Step 3 — Explore the Data

In [ ]:
# Check for missing values
print('Missing values:', df.isnull().sum().sum())
print('\nBasic statistics:')
df.describe().round(1)

In [ ]:
# Chart 1: Average subject marks per career
subjects = ['Math', 'Physics', 'Chemistry', 'Biology', 'English']
avg = df.groupby('Career_Label')[subjects].mean()

fig, ax = plt.subplots(figsize=(12, 5))
avg.T.plot(kind='bar', ax=ax, colormap='tab10', width=0.8)
ax.set_title('Average subject marks per career path', fontsize=14)
ax.set_xlabel('Subject')
ax.set_ylabel('Average mark (0-100)')
ax.legend(loc='lower right', fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Sport skill level per career
fig, ax = plt.subplots(figsize=(10, 4))
order = df.groupby('Career_Label')['Sport_Skill_Level'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Career_Label', y='Sport_Skill_Level', order=order, palette='Set2', ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right', fontsize=9)
ax.set_title('Sport skill level by career path')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

## Step 4 — Train the Random Forest Model

In [ ]:
# Define features (inputs) and label (output)
FEATURES = ['Math', 'Physics', 'Chemistry', 'Biology', 'English',
            'Attendance_Pct', 'Focus_Duration_Min',
            'Sport_Type_Enc', 'Sport_Skill_Level']

X = df[FEATURES]         # 9 input columns
y = df['Career_Label']   # 1 output column

# Split: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'Training set: {X_train.shape[0]} students')
print(f'Testing set:  {X_test.shape[0]} students')

In [ ]:
# Train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate accuracy
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Model Accuracy: {acc*100:.1f}%')
print()
print(classification_report(y_test, y_pred))

In [ ]:
# Chart 3: Feature importance — which input matters most?
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Feature importance — what the model relies on most')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.show()

## Step 5 — Predict Career for a New Student
### Edit the values below to match your own profile!

In [ ]:
# ==============================================================
#   CHANGE THESE VALUES TO YOUR OWN MARKS AND SPORTS PROFILE
# ==============================================================
# Sport encoding: Football=0, Cricket=1, Swimming=2,
#                 Athletics=3, Basketball=4, Badminton=5, None=6

my_profile = {
    'Math':               85,   # Your Math marks (0-100)
    'Physics':            78,   # Your Physics marks
    'Chemistry':          62,   # Your Chemistry marks
    'Biology':            64,   # Your Biology marks
    'English':            70,   # Your English marks
    'Attendance_Pct':     82,   # Your attendance %
    'Focus_Duration_Min': 58,   # How long you study daily (minutes)
    'Sport_Type_Enc':      0,   # 0 = Football (see encoding above)
    'Sport_Skill_Level':   8    # Your sport skill (1-10)
}
# ==============================================================

student_df = pd.DataFrame([my_profile])
proba      = model.predict_proba(student_df)[0]
careers    = model.classes_

results = sorted(zip(careers, proba), key=lambda x: x[1], reverse=True)

print('Top career recommendations for you:')
print('-' * 45)
for rank, (career, prob) in enumerate(results, 1):
    bar = '#' * int(prob * 40)
    print(f'  {rank}. {career:<30} {prob*100:5.1f}%  {bar}')

## Step 6 — Visualize Your Results

In [ ]:
labels = [r[0] for r in results]
values = [r[1] * 100 for r in results]
colors = ['#534AB7', '#185FA5', '#854F0B', '#0F6E56', '#993C1D', '#3B6D11']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(labels[::-1], values[::-1], color=colors[::-1], height=0.6)

for bar, val in zip(bars, values[::-1]):
    ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

ax.set_xlim(0, 105)
ax.set_xlabel('Probability (%)', fontsize=11)
ax.set_title('Your personalised career recommendations', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('career_prediction_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as career_prediction_result.png')

## Bonus — Save the trained model for reuse

In [ ]:
import pickle

# Save model to file
with open('career_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved as career_model.pkl')

# To load it back later:
# with open('career_model.pkl', 'rb') as f:
#     loaded_model = pickle.load(f)
print('Done! Your complete Career Counselor ML project is ready.')